# Workflows: one command rebuilds the result

## Learning objectives

- Turn a README's "run these in this order" into a declared workflow, where each step names what it reads and what it writes.
- Let the tool decide what's stale, so a changed parameter reruns only the step that reads it.
- Delete every output and get the committed result back from the raw data with one command.

## The problem

- The workshop's exercises use one project, a study of Palmer penguins, and add a capability in each exercise.
- `project` computes one table from the 2007 field season: how much heavier a male penguin is than a female of the same species, in grams. Exercise 1 did it in one script. Here it's three: `clean.py`, `validate.py` and `report.py`.
- The README says to run the three in order, and after a change to `params.yaml` to "rerun the steps that need it".
- Nothing records which step reads what. A person has to know, every time.

## What you'll do

1. Step 1: write a Snakefile that declares the three steps, rebuild from the raw data, and commit it.
2. Step 2: change a parameter, and watch Snakemake rerun one step.
3. Run the check.

- The exercise takes about 15 to 20 minutes.
- Only reading? Each step shows the output and the fix.
- [Exercise 2 in Nextflow](nextflow.md) is the same exercise in Nextflow, on your own machine.

## The rule

- The document and the data are the pets. Everything that runs them is cattle.
- Here the pets are the data, the scripts, `params.yaml` and the Snakefile, all in Git. The cattle are `results/` and `.snakemake/`. Delete them, and the workflow builds them again.
- `metrics.json` is committed, so a reader sees the numbers without running anything. It has to come back byte for byte from the raw data.

## Get a terminal

- You read this page and paste commands into a terminal. Nothing runs on the page.
- The output shown here is recorded from the untouched project, so it stops at the failures. The fixes are yours.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-02-workflows?urlpath=lab/workspaces/exercise)**

- Free, with nothing to install. The first start can take a few minutes.
- The terminal is on the left, already in the `project` folder. This page is on the right.
- Copy a command: select it and press Ctrl+C, or Cmd+C on a Mac.
- Paste it in the terminal: Ctrl+Shift+V, or right-click and choose Paste. Cmd+V on a Mac is untested, so right-click if it doesn't work.
- Edit a file: double-click it in the file panel on the far left. Save with Ctrl+S.
- A session is temporary and has 2 GB of memory. Download anything you want to keep.

**Or use your own terminal**

- Optional: [use uv on your own machine](../../uv.md).

- Linux or macOS, on x86_64 or ARM, or Windows through WSL2. On Windows, keep the folder in your WSL2 home, not under `/mnt/c`.
- You need [conda](https://github.com/conda-forge/miniforge). You don't need Jupyter.
- Paste this once:

```bash
git clone --branch binder-02-workflows --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git workflows
cd workflows
conda env create --quiet --file .binder/environment.yml
conda activate repro-workflows
python scripts/setup_project.py
cd project
```

- Every command below runs in that `project` folder.
- To start over, go to the folder above it and run `python scripts/setup_project.py --reset`. This deletes your work.

## The project

- `project` has the 2007 penguins, three scripts, `params.yaml` and a committed `metrics.json`, in one commit.
- The README is the only record of which step reads what, and in which order.

```bash
cat README.md
git log --format='%h  %s'
cat params.yaml
cat metrics.json
```

In [1]:
# Recorded output. You don't need to run this.
import sys

sys.path.insert(0, ".exercise")
from page import reset, run

run("cat README.md", "git log --format='%h  %s'", "cat params.yaml", "cat metrics.json")

$ cat README.md
# Sexual dimorphism in Palmer penguins

How much heavier is a male penguin than a female of the same species? We report the difference of the mean body masses, in grams, per species, for the 2007 field season.

Run clean.py, validate.py, report.py in this order:

1. `python clean.py` drops penguins with no body mass or no sex, applies the repairs `params.yaml` turns on, and writes `results/clean.csv`.
2. `python validate.py` checks that the columns and rows are there, and writes `results/validated.csv`.
3. `python report.py` computes the difference per species with the statistic `params.yaml` names, and writes `metrics.json`.

After a change to `params.yaml`, rerun the steps that need it.

Data: palmerpenguins (Horst, Hill and Gorman 2020, doi:10.5281/zenodo.3960218), CC0, from Gorman, Williams and Fraser (2014), PLOS ONE 9(3): e90081. Collected by Palmer Station LTER with NSF support.

$ git log --format='%h  %s'
750eec1  Add the 2007 penguins, the three scripts and th

## Step 1: declare the steps

**Problem:** the README gives an order, and that's the whole record. Run it.

```bash
python clean.py && python validate.py && python report.py
git status --short
```

In [2]:
# Recorded output. You don't need to run this.
run("python clean.py && python validate.py && python report.py", "git status --short")

$ python clean.py && python validate.py && python report.py
clean: kept 103 of 110 penguins
validate: 103 rows pass
{
  "dimorphism_g_adelie": 648.9,
  "dimorphism_g_chinstrap": 250.0,
  "dimorphism_g_gentoo": 934.2,
  "n_rows": 103
}

$ git status --short


**Facts**

- `git status --short` prints nothing: the committed numbers are current. You know that only because you ran all three steps again. After a change to `params.yaml` or a script, the README leaves it to you to know which steps to rerun.
- A README order is a procedure. A Snakefile is a declaration. Each rule names its input files, its output files and the command. Snakemake builds the graph of steps from the file names, and it reruns a rule when an output is missing, an input is newer than its output, or the rule's parameters changed.
- `configfile: "params.yaml"` loads the parameters into `config`. A rule's `params:` line names the parameters its script reads. Snakemake records them with the output, and a change reruns the rule. The scripts still read `params.yaml` themselves. The Snakefile declares who depends on what.
- The first rule is the default target. A rule named `all`, with the final files as its inputs, is the convention.
- `snakemake -n` is a dry run. It lists what would run and runs nothing. `snakemake --cores 1` runs it.
- `--forceall` runs every rule, whatever the file dates say. Use it once, when you adopt a workflow, because nobody can vouch for the files already there.
- The project's `profiles/default/config.yaml` sets the greedy scheduler. It's fine for a small workflow, and it avoids a solver that isn't available on every machine. Because of it, Snakemake prints `Using workflow specific profile profiles/default` first.
- `results/` and `.snakemake/` are in `.gitignore`. `metrics.json` is not.

**Steps**

1. Create a file named `Snakefile` in the `project` folder with this, and add a rule for `validate.py` and a rule for `report.py` in the same form. `validate.py` reads `results/clean.csv` and writes `results/validated.csv`. `report.py` reads `results/validated.csv` and the `report` section of `params.yaml`, and writes `metrics.json`. Indent with four spaces.

```text
configfile: "params.yaml"

rule all:
    input: "metrics.json"

rule clean:
    input: "data/penguins.csv", "clean.py"
    output: "results/clean.csv"
    params: config["clean"]
    shell: "python clean.py"
```

2. Ask Snakemake what it would do. It should say there's nothing to do, and that three rules have no record of how their outputs were made. If it reports an error instead, read the line number it names.

```bash
snakemake -n
```

3. Build everything from the raw data once, and look at the result. The diff is empty, because the committed numbers were current, and the dry run now says nothing to do, with no missing records.

```bash
snakemake --cores 1 --forceall
git diff metrics.json
snakemake -n
```

4. Commit the Snakefile.

```bash
git add Snakefile
git commit -m "Declare the three steps"
```

## Step 2: change a parameter

**Problem:** `params.yaml` says `statistic: mean`. Change it to `median`, and which steps have to rerun? With the README, you decide. With the Snakefile, you ask.

**Facts**

- Only `report` declares `config["report"]`, so a change to `statistic` reruns `report` and nothing else. `clean` declares `config["clean"]`, which didn't change.
- Had you listed `params.yaml` as an input instead, any edit to it would rerun `clean` and every step after it. Declare what each step reads, no more.
- A decision such as which statistic to report lives in `params.yaml`, in Git, not in a command somebody typed.

**Steps**

1. Open `params.yaml`, change `statistic: mean` to `statistic: median`, and save.
2. Ask what would run, then run it, and read the new numbers.

```bash
snakemake -n
snakemake --cores 1
cat metrics.json
```

- The dry run's job table lists `report` and `all`, and the reason for `report` is that its params changed. The new numbers are the medians:

```text
{
  "dimorphism_g_adelie": 625.0,
  "dimorphism_g_chinstrap": 200.0,
  "dimorphism_g_gentoo": 925.0,
  "n_rows": 103
}
```

3. Change it back to `statistic: mean`, save, and run again. `report` reruns once more, and `git status --short` prints nothing: `metrics.json` is the committed file again, byte for byte.

```bash
snakemake --cores 1
git status --short
```

## Check your work

- Run this when both steps are done. Each line says what to do next.
- It clones your last commit into a throwaway folder, deletes every output and builds them with your Snakefile. Then it compares the result with the one you committed, changes `report.statistic`, and checks that `report` reruns and `clean` doesn't.
- It takes a few seconds. `--short` runs the same parts.

```bash
python ../check.py
```

On the starter project it fails like this:

In [3]:
# Recorded output. You don't need to run this.
reset()
run("python ../check.py")

$ python ../check.py
FAIL  No Snakefile is committed. Write the rules in step 1, then git add Snakefile and commit.

0 of 1 parts pass.


## Summary

- A README order is a procedure. A Snakefile is a declaration: each step names what it reads and what it writes, and the tool knows what's stale.
- Declare the parameters a step reads, so a change reruns that step and no other.
- If you can delete every output and get them back with one command, the project holds the result.
- Exercise 3 does the same thing with DVC stages, and versions the data too.
- Nextflow does what Snakemake does. [Exercise 2 in Nextflow](nextflow.md) is this exercise with a `main.nf`.

## On your own project

- Find the README, script or notebook cell that says "run these in this order". Write it as rules.
- Add every file a step reads to its inputs, and every parameter it reads to its params.
- Delete every output and rebuild. Commit the ones a reader needs.

## About the data

- `data/penguins.csv` is the 2007 field season of the Palmer penguins, 110 penguins, unchanged from the [palmerpenguins](https://allisonhorst.github.io/palmerpenguins/) package. 103 of them have a body mass and a sex.
- Data: palmerpenguins (Horst, Hill and Gorman 2020, [doi:10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218)), CC0, from Gorman, Williams and Fraser (2014), PLOS ONE 9(3): e90081. Collected by Palmer Station LTER with NSF support.